# Анализ ДТП 2022–2024  
## Исполнитель 1: Классификаторы, факторы риска, витрины DataLens

**Структура ноутбука:**
1. Импорты и настройки
2. Загрузка и фильтрация данных (2022–2024)
3. Укрупнённый классификатор типов ДТП
4. Укрупнённый классификатор нарушений ПДД
5. Укрупнённый классификатор дорожных дефектов
6. Аналитические фичи: человек, дорога, ТС, время
7. Реестр факторов риска (factor_registry)
8. Витрины DataLens (mart_*)
9. **Блок исполнителя 2**: пространственные признаки (feat_spatial_dtp → mart_spatial_risk)
10. **Блок исполнителя 3 [ЗАГЛУШКА]**: погода и световая среда (feat_weather_dtp → mart_weather_context)

---
## 0. Импорты и настройки

In [49]:
import pandas as pd
import numpy as np
import ast
import re
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# ─── Пути к файлам — замените на свои ────────────────────────────────────────
PATH_DTP          = 'output/fact_dtp.csv'
PATH_PARTICIPANT  = 'output/fact_participant.csv'
PATH_VEHICLE      = 'output/fact_vehicle.csv'

# От исполнителя 2 (после получения):
PATH_SPATIAL      = 'output_spatial/feat_spatial_dtp.csv'
PATH_CELL_SPACE   = 'output_spatial/agg_cell_space.csv'

# От исполнителя 3 (после получения):
PATH_WEATHER      = 'foutput_w/feat_weather_dtp.csv'

# Период анализа
YEAR_FROM = 2022
YEAR_TO   = 2024

---
## 1. Загрузка и фильтрация данных

In [50]:
# ── fact_dtp ──────────────────────────────────────────────────────────────────
dtp = pd.read_csv(PATH_DTP, low_memory=False)
dtp['moment_date'] = pd.to_datetime(dtp['moment_date'], errors='coerce')
dtp['year'] = dtp['moment_date'].dt.year
dtp = dtp[dtp['year'].between(YEAR_FROM, YEAR_TO)].copy()

print(f'fact_dtp после фильтрации: {len(dtp):,} строк')
print('По годам:')
print(dtp['year'].value_counts().sort_index())

fact_dtp после фильтрации: 24,553 строк
По годам:
year
2022    7629
2023    7977
2024    8947
Name: count, dtype: int64


In [51]:
# ── fact_participant ──────────────────────────────────────────────────────────
part = pd.read_csv(PATH_PARTICIPANT, low_memory=False)
part = part[part['dtp_id'].isin(dtp['dtp_id'])].copy()
print(f'fact_participant: {len(part):,} строк')

# Числовые поля
part['person_age']            = pd.to_numeric(part['person_age'], errors='coerce')
part['driver_service_length'] = pd.to_numeric(part['driver_service_length'], errors='coerce')

fact_participant: 58,359 строк


In [52]:
# ── fact_vehicle ──────────────────────────────────────────────────────────────
veh = pd.read_csv(PATH_VEHICLE, low_memory=False)
veh = veh[veh['dtp_id'].isin(dtp['dtp_id'])].copy()
print(f'fact_vehicle: {len(veh):,} строк')

veh['manufacture_year'] = pd.to_numeric(veh['manufacture_year'], errors='coerce')

fact_vehicle: 42,340 строк


In [53]:
# ── Вспомогательная функция: парсинг строки-списка Python ─────────────────────
def parse_list_field(val):
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return []
    try:
        result = ast.literal_eval(str(val))
        if isinstance(result, list):
            return [str(x).strip() for x in result]
    except Exception:
        pass
    cleaned = re.sub(r'^\[|\]$', '', str(val))
    return [x.strip().strip("'\" ") for x in cleaned.split(',') if x.strip()]

---
## 2. Укрупнённый классификатор типов ДТП  
Справочник `type_code` → `type_name_decoded` → `type_group`

In [54]:
TYPE_CODE_DICT = {
    1:  'Столкновение ТС',
    2:  'Опрокидывание',
    3:  'Наезд на стоящее ТС',
    4:  'Наезд на препятствие',
    5:  'Наезд на пешехода',
    6:  'Конфликт с велотранспортом',
    8:  'Наезд на животное',
    9:  'Падение пассажира',
    91: 'Наезд на пешехода (вне проезжей части)',
    94: 'Наезд на препятствие (иное)',
    95: 'Конфликт с СИМ (самокаты/моноколеса)',
    96: 'Конфликт с СИМ (самокаты/моноколеса)',
    0:  'Не установлено',
}

# Укрупнение → 6 аналитических групп
TYPE_GROUP_MAP = {
    1:  'Столкновение',
    2:  'Опрокидывание',
    3:  'Наезд на ТС',
    4:  'Наезд на препятствие',
    5:  'Пешеходный конфликт',
    6:  'Велосипедный / СИМ конфликт',
    8:  'Прочее',
    9:  'Прочее',
    91: 'Пешеходный конфликт',
    94: 'Наезд на препятствие',
    95: 'Велосипедный / СИМ конфликт',
    96: 'Велосипедный / СИМ конфликт',
    0:  'Не установлено',
}

dtp['type_name_decoded'] = dtp['type_code'].map(TYPE_CODE_DICT).fillna('Неизвестный код')
dtp['type_group']        = dtp['type_code'].map(TYPE_GROUP_MAP).fillna('Прочее')

print('Распределение по группам типов ДТП:')
print(dtp['type_group'].value_counts())

Распределение по группам типов ДТП:
type_group
Столкновение                   10999
Пешеходный конфликт             8212
Велосипедный / СИМ конфликт     1390
Прочее                          1312
Наезд на препятствие            1175
Наезд на ТС                      863
Опрокидывание                    601
Не установлено                     1
Name: count, dtype: int64


---
## 3. Укрупнённый классификатор нарушений ПДД  
Поля: `main_pdd_derangements`, `attendant_pdd_derangements` в `fact_participant`

In [55]:
part['main_viol_list']      = part['main_pdd_derangements'].apply(parse_list_field)
part['attendant_viol_list'] = part['attendant_pdd_derangements'].apply(parse_list_field)

print('Топ-20 основных нарушений ПДД:')
all_main_viols = [v for lst in part['main_viol_list'] for v in lst]
for viol, cnt in Counter(all_main_viols).most_common(20):
    print(f'  {cnt:6,}  {viol[:80]}')

Топ-20 основных нарушений ПДД:
  33,292  Не нарушал ПДД
   4,254  Неправильный выбор дистанции
   3,516  Несоблюдение очередности проезда
   3,131  Несоответствие скорости конкретным условиям движения
   2,511  Нарушение правил проезда пешеходного перехода
   1,979  Непредоставление преимущества в движении пешеходу
   1,575  Нарушение правил перестроения
   1,271  Переход через проезжую часть вне пешеходного перехода в зоне его видимости либо 
   1,234  Нарушение требований сигналов светофора
     863  Несоблюдение условий, разрешающих движение транспорта задним ходом
     572  Нарушение правил расположения ТС на проезжей части
     561  Нарушения правил пользования общественным транспортом
     434  Неподчинение сигналам регулирования
     312  Нарушение правил перевозки людей
     304  Пересечение велосипедистом проезжей части по пешеходному переходу
     302  Невыполнение требований обеспечения безопасности при начале движения
     252  Несоблюдение бокового интервала
     239  Пере

In [56]:
# Классификатор: ключевые слова → группа нарушения
PDD_GROUP_RULES = [
    ('Скорость',                 ['превышение установленной скорости', 'скорост', 'несоответствующей скорост']),
    ('Приоритет / светофор',     ['преимущественного проезда', 'не уступил', 'уступить', 'главной дороги',
                                   'красный', 'светофор', 'сигналу регулировщика', 'регулирован']),
    ('Пешеходный переход',       ['пешеход', 'пешеходный переход']),
    ('Алкоголь / состояние',     ['алкоголь', 'наркотик', 'опьянен', 'болезн', 'усталост']),
    ('Манёвр',                   ['перестроен', 'поворот', 'разворот', 'обгон', 'опережен']),
    ('Дистанция / интервал',     ['дистанц', 'интервал']),
    ('Техническое состояние ТС', ['техническ', 'неисправн']),
    ('Не нарушал',               ['не нарушал']),
    ('Прочие нарушения',         ['иные', 'прочие', 'другие']),
]

def classify_pdd_violation(viol_text):
    text_lower = viol_text.lower()
    for group_name, keywords in PDD_GROUP_RULES:
        if any(kw in text_lower for kw in keywords):
            return group_name
    return 'Прочее'

# Классификация основных и сопутствующих нарушений
part['main_viol_groups']      = part['main_viol_list'].apply(
    lambda lst: list(set(classify_pdd_violation(v) for v in lst)) if lst else [])
part['attendant_viol_groups'] = part['attendant_viol_list'].apply(
    lambda lst: list(set(classify_pdd_violation(v) for v in lst)) if lst else [])

# Агрегат нарушений на уровень ДТП (основные)
dtp_viol = (
    part.explode('main_viol_groups')
    .groupby('dtp_id')['main_viol_groups']
    .apply(lambda x: list(set(x.dropna())))
    .reset_index(name='dtp_viol_groups')
)
dtp = dtp.merge(dtp_viol, on='dtp_id', how='left')
dtp['dtp_viol_groups'] = dtp['dtp_viol_groups'].apply(lambda x: x if isinstance(x, list) else [])

print('Распределение групп нарушений по участникам:')
for g, c in Counter([v for lst in part['main_viol_groups'] for v in lst]).most_common():
    print(f'  {c:7,}  {g}')

Распределение групп нарушений по участникам:
   33,292  Не нарушал
    7,309  Прочее
    6,145  Пешеходный переход
    4,506  Дистанция / интервал
    3,139  Скорость
    1,668  Приоритет / светофор
    1,655  Манёвр
       30  Прочие нарушения
       27  Техническое состояние ТС


---
## 4. Укрупнённый классификатор дорожных дефектов  
Поле `road_drawbacks` в `fact_dtp`.  
Все уникальные значения из датасета учтены ниже.

In [57]:
dtp['road_drawbacks_list'] = dtp['road_drawbacks'].apply(parse_list_field)

# Укрупнённый классификатор — 10 групп
ROAD_DEFECT_RULES = [
    ('Нет недостатков',        ['отсутствие перечисленных недостатков']),
    ('Разметка',               ['горизонтальной разметки', 'дорожной разметки']),
    ('Дорожные знаки',         ['дорожных знаков', 'видимость дорожных знаков']),
    ('Освещение',              ['освещени']),
    ('Светофор',               ['светофор']),
    ('Дефекты покрытия',       ['дефекты покрытия', 'выбоины', 'просадки', 'повреждения покрытия']),
    ('Зимнее содержание',      ['зимнего содержания', 'снег', 'снежный накат']),
    ('Дорожные работы (ТСОД)', ['временных тсод', 'место производства работ']),
    ('Люки / канализация',     ['люков смотровых', 'ливневой канализации']),
    ('Прочие недостатки',      ['иные недостатки']),
]

def classify_road_defect(defect_text):
    text_lower = defect_text.lower()
    for group_name, keywords in ROAD_DEFECT_RULES:
        if any(kw in text_lower for kw in keywords):
            return group_name
    return 'Прочее'

dtp['road_defect_groups'] = dtp['road_drawbacks_list'].apply(
    lambda lst: list(set(classify_road_defect(d) for d in lst)) if lst else ['Нет данных'])

print('Распределение групп дорожных дефектов:')
for defect, cnt in Counter([d for lst in dtp['road_defect_groups'] for d in lst]).most_common():
    print(f'  {cnt:7,}  {defect}')

Распределение групп дорожных дефектов:
   22,203  Нет недостатков
    1,011  Дорожные знаки
      838  Разметка
      327  Нет данных
      140  Прочие недостатки
       67  Зимнее содержание
       63  Дорожные работы (ТСОД)
       58  Светофор
       50  Прочее
       26  Дефекты покрытия
       23  Освещение
        6  Люки / канализация


---
## 5. Аналитические фичи — факторы риска

### 5.1 Блок «Человек»

In [58]:
# ── Флаги нарушений на уровне участника ──────────────────────────────────────
def has_kw(lst, keywords):
    joined = ' '.join(lst).lower()
    return any(kw in joined for kw in keywords)

part['has_speed_viol']    = part['main_viol_list'].apply(lambda l: has_kw(l, ['скорост', 'превышен']))
part['has_priority_viol'] = part['main_viol_list'].apply(lambda l: has_kw(l, ['преимущественного', 'не уступил', 'светофор', 'регулирован', 'красный']))
# part['has_alcohol_viol']  = part['main_viol_list'].apply(lambda l: has_kw(l, ['алкоголь', 'опьянен', 'наркотик']))
part['is_pedestrian']     = part['part_type_code'].str.lower().str.contains('пешеход', na=False)
part['is_driver']         = part['part_type_code'].str.lower().str.contains('водитель', na=False)

# Агрегация на уровень ДТП
dtp_human = (
    part.groupby('dtp_id').agg(
        has_speed_violation     = ('has_speed_viol',    'any'),
        has_priority_violation  = ('has_priority_viol', 'any'),
       
        has_pedestrian_conflict = ('is_pedestrian',     'any'),
        driver_age_min          = ('person_age',        lambda x: x[part.loc[x.index,'is_driver']].min() if part.loc[x.index,'is_driver'].any() else np.nan),
        participant_count       = ('participant_id',    'count'),
        service_length_min      = ('driver_service_length', 'min'),
    ).reset_index()
)

dtp_human['has_young_driver']     = dtp_human['driver_age_min'] < 25
dtp_human['has_novice_driver']    = dtp_human['service_length_min'] < 2  # стаж < 2 лет
dtp = dtp.merge(dtp_human, on='dtp_id', how='left')

print('Флаги «Человек»:')
for col in ['has_speed_violation','has_priority_violation','has_alcohol_violation',
            'has_pedestrian_conflict','has_young_driver','has_novice_driver']:
    if col in dtp.columns:
        print(f'  {col}: {int(dtp[col].sum()):,} ДТП')

Флаги «Человек»:
  has_speed_violation: 3,137 ДТП
  has_priority_violation: 1,660 ДТП
  has_pedestrian_conflict: 8,508 ДТП
  has_young_driver: 5,355 ДТП
  has_novice_driver: 2,183 ДТП


### 5.2 Блок «Транспортное средство»

In [59]:
# ── Признаки ТС ──────────────────────────────────────────────────────────────
# Возраст ТС
 
 

# Тип шин (зимние / летние / всесезонные)
veh['tyre_group'] = veh['tyre_type_code'].fillna('Не указано').apply(
    lambda x: 'Зимние шипованные' if 'шипован' in str(x).lower()
    else ('Зимние нешипованные' if 'нешипован' in str(x).lower()
    else ('Летние' if 'летни' in str(x).lower()
    else ('Всесезонные' if 'всесезон' in str(x).lower() else 'Не указано')))
)

# Укрупнённый тип ТС
def transport_class(name):
    name = str(name).lower()
    if 'мотоцикл' in name or 'мопед' in name:     return 'Мотоцикл / мопед'
    if 'грузов' in name:                            return 'Грузовой'
    if 'автобус' in name or 'микроавтобус' in name: return 'Автобус'
    if 'такси' in name:                             return 'Такси'
    if 'легков' in name or '-класс' in name:        return 'Легковой'
    return 'Прочее'

veh['transport_class'] = veh['transport_type_name'].apply(transport_class)

# Агрегация на уровень ДТП
veh_damage_list = veh.groupby('dtp_id')['damage_dispositions'].apply(
    lambda x: [v for lst in x.dropna().apply(parse_list_field) for v in lst]
).reset_index(name='damage_list')

dtp_vehicle = (
    veh.groupby('dtp_id').agg(
    
        has_winter_tyres     = ('tyre_group',  lambda x: (x.str.contains('Зимние', na=False)).any()),
        transport_classes    = ('transport_class', lambda x: list(set(x))),
        has_truck            = ('transport_class', lambda x: 'Грузовой' in x.values),
        has_bus              = ('transport_class', lambda x: 'Автобус' in x.values),
        has_taxi             = ('transport_class', lambda x: 'Такси' in x.values),
        has_motorcycle       = ('transport_class', lambda x: 'Мотоцикл / мопед' in x.values),
    ).reset_index()
)
 

dtp = dtp.merge(dtp_vehicle, on='dtp_id', how='left')
dtp = dtp.merge(veh_damage_list, on='dtp_id', how='left')

print('Флаги «ТС»:')
for col in ['has_winter_tyres','has_truck','has_bus','has_taxi','has_motorcycle']:
    if col in dtp.columns:
        print(f'  {col}: {int(dtp[col].sum()):,} ДТП')

Флаги «ТС»:
  has_winter_tyres: 2,778 ДТП
  has_truck: 2,104 ДТП
  has_bus: 2,063 ДТП
  has_taxi: 6,063 ДТП
  has_motorcycle: 1,965 ДТП


### 5.3 Блок «Дорога»

In [60]:
# ── Флаг плохого состояния дороги ────────────────────────────────────────────
BAD_ROAD_GROUPS = {
    'Разметка', 'Дорожные знаки', 'Освещение', 'Светофор',
    'Дефекты покрытия', 'Зимнее содержание', 'Дорожные работы (ТСОД)',
    'Люки / канализация', 'Прочие недостатки', 'Прочее'
}
dtp['has_bad_road_condition'] = dtp['road_defect_groups'].apply(
    lambda lst: bool(set(lst) & BAD_ROAD_GROUPS))

# ── Геометрия дороги (поля из fact_dtp) ──────────────────────────────────────
for col in ['traffic_lane_amount', 'dtp_traffic_lane', 'traffic_area_width',
            'sidewalk_width', 'wayside_width']:
    if col in dtp.columns:
        dtp[col] = pd.to_numeric(dtp[col], errors='coerce')

# Флаг многополосной дороги
if 'traffic_lane_amount' in dtp.columns:
    dtp['is_multilane'] = dtp['traffic_lane_amount'] >= 4

# Тип поверхности (srf_code) — если есть
if 'srf_code' in dtp.columns:
    dtp['road_surface_type'] = dtp['srf_code'].fillna('Не указано')

print('Флаги «Дорога»:')
for col in ['has_bad_road_condition', 'is_multilane']:
    if col in dtp.columns:
        print(f'  {col}: {int(dtp[col].sum()):,} ДТП')

Флаги «Дорога»:
  has_bad_road_condition: 2,023 ДТП
  is_multilane: 11,189 ДТП


### 5.4 Блок «Время»

In [61]:
# ── Временные признаки ───────────────────────────────────────────────────────
dtp['hour']        = pd.to_datetime(dtp['moment_time'], format='%H:%M:%S', errors='coerce').dt.hour
dtp['month']       = dtp['moment_date'].dt.month
dtp['day_of_week'] = dtp['moment_date'].dt.dayofweek  # 0=Пн, 6=Вс
dtp['is_weekend']  = dtp['day_of_week'] >= 5

def time_of_day(h):
    if pd.isna(h): return 'Неизвестно'
    if 6  <= h < 12: return 'Утро (6–12)'
    if 12 <= h < 18: return 'День (12–18)'
    if 18 <= h < 22: return 'Вечер (18–22)'
    return 'Ночь (22–6)'

def get_season(m):
    if pd.isna(m): return 'Неизвестно'
    if m in [12, 1, 2]: return 'Зима'
    if m in [3, 4, 5]:  return 'Весна'
    if m in [6, 7, 8]:  return 'Лето'
    return 'Осень'

dtp['time_of_day'] = dtp['hour'].apply(time_of_day)
dtp['season']      = dtp['month'].apply(get_season)
dtp['is_night']    = dtp['hour'].apply(lambda h: bool(h < 6 or h >= 22) if not pd.isna(h) else False)
dtp['is_rush_hour']= dtp['hour'].apply(lambda h: bool(h in range(7,10) or h in range(17,20)) if not pd.isna(h) else False)

print('Временные признаки:')
print(dtp['time_of_day'].value_counts())
print()
print(dtp['season'].value_counts())

Временные признаки:
time_of_day
День (12–18)     8981
Вечер (18–22)    6285
Утро (6–12)      5654
Ночь (22–6)      3633
Name: count, dtype: int64

season
Лето     6874
Осень    6764
Весна    6131
Зима     4784
Name: count, dtype: int64


### 5.5 Тяжесть последствий

In [62]:
# ── Тяжесть ──────────────────────────────────────────────────────────────────
for col in ['dead_count', 'injured_count', 'dead_children_count', 'injured_children_count']:
    if col in dtp.columns:
        dtp[col] = pd.to_numeric(dtp[col], errors='coerce').fillna(0).astype(int)

def severity_level(row):
    if row.get('dead_count', 0) > 0:      return 'Гибель'
    if row.get('injured_count', 0) > 0:   return 'Ранение'
    return 'Без пострадавших'

dtp['severity'] = dtp.apply(severity_level, axis=1)

# hv_type_code — тип пострадавшего участника, агрегируем из fact_participant
hv_agg = (
    part.groupby('dtp_id')['hv_type_code']
    .apply(lambda x: list(x.dropna().unique()))
    .reset_index(name='hv_types')
)
dtp = dtp.merge(hv_agg, on='dtp_id', how='left')

print('Тяжесть последствий:')
print(dtp['severity'].value_counts())

Тяжесть последствий:
severity
Ранение    23656
Гибель       897
Name: count, dtype: int64


---
## 6. Реестр факторов риска (factor_registry)

Разделение:
- **прямой** — берётся из сырого поля без преобразований
- **производный** — вычислен из одного или нескольких сырых полей
- **обогащённый** — пришёл от исполнителей 2 или 3

---
## 7. Витрины DataLens

### 7.1 mart_dtp_overview

In [63]:
overview_cols = [
    'dtp_id', 'moment_date', 'year', 'month', 'day_of_week', 'hour',
    'time_of_day', 'season', 'is_weekend', 'is_night', 'is_rush_hour',
    'type_code', 'type_name_decoded', 'type_group',
    'severity', 'dead_count', 'injured_count',
    'dead_children_count', 'injured_children_count',
    'vehicles_count', 'participant_count',
    'area_id', 'district_id', 'source_year',
]
# Добавляем пространственные поля если есть
for col in ['place_latitude', 'place_longitude', 'road_type_code',
            'street_name', 'road_name', 'road_loc',
            'traffic_lane_amount', 'srf_code', 'road_sign_code']:
    if col in dtp.columns:
        overview_cols.append(col)

mart_dtp_overview = dtp[[c for c in overview_cols if c in dtp.columns]].copy()
mart_dtp_overview.to_csv('mart_dtp_overview.csv', index=False, encoding='utf-8-sig')
print(f'mart_dtp_overview: {len(mart_dtp_overview):,} строк × {mart_dtp_overview.shape[1]} колонок')

mart_dtp_overview: 24,553 строк × 33 колонок


### 7.2 mart_time_dynamics

In [65]:
mart_time_dynamics = (
    dtp.groupby(['year', 'month', 'season', 'day_of_week', 'time_of_day', 'is_weekend'])
    .agg(
        dtp_count               = ('dtp_id', 'count'),
        dead_total              = ('dead_count', 'sum'),
        injured_total           = ('injured_count', 'sum'),
        dead_children_total     = ('dead_children_count', 'sum'),
        injured_children_total  = ('injured_children_count', 'sum'),
        pct_speed_viol          = ('has_speed_violation', 'mean'),
        pct_priority_viol       = ('has_priority_violation', 'mean'),

        pct_bad_road            = ('has_bad_road_condition', 'mean'),
        pct_pedestrian          = ('has_pedestrian_conflict', 'mean'),
        pct_night               = ('is_night', 'mean'),
        pct_severity_fatal      = ('severity', lambda x: (x == 'Гибель').mean()),
    )
    .reset_index()
)

mart_time_dynamics.to_csv('mart_time_dynamics.csv', index=False, encoding='utf-8-sig')
print(f'mart_time_dynamics: {len(mart_time_dynamics):,} строк')

mart_time_dynamics: 1,008 строк


### 7.3 mart_factor_profile

In [66]:
factor_cols = [
    'dtp_id', 'year', 'type_group', 'severity',
    'has_speed_violation', 'has_priority_violation', 'has_alcohol_violation',
    'has_pedestrian_conflict', 'has_bad_road_condition',
    'has_young_driver', 'has_novice_driver',
     'has_truck', 'has_bus', 'has_taxi', 'has_motorcycle',
    'is_night', 'is_weekend', 'is_rush_hour',
    'time_of_day', 'season',
    'dead_count', 'injured_count', 'participant_count',
]

mart_factor_profile = dtp[[c for c in factor_cols if c in dtp.columns]].copy()

# Булевы → Да/Нет для DataLens-фильтров
bool_cols = [c for c in mart_factor_profile.columns
             if mart_factor_profile[c].dtype == bool or mart_factor_profile[c].isin([True, False]).all()]
for col in bool_cols:
    mart_factor_profile[col] = mart_factor_profile[col].map({True: 'Да', False: 'Нет'})

mart_factor_profile.to_csv('mart_factor_profile.csv', index=False, encoding='utf-8-sig')
print(f'mart_factor_profile: {len(mart_factor_profile):,} строк × {mart_factor_profile.shape[1]} колонок')

mart_factor_profile: 24,553 строк × 22 колонок


### 7.4 mart_spatial_risk  
> Базовая агрегация по `district_id`.  


In [68]:

feat_spatial = pd.read_csv(PATH_SPATIAL, low_memory=False)
agg_cell     = pd.read_csv(PATH_CELL_SPACE, low_memory=False)

print(f'feat_spatial_dtp: {len(feat_spatial):,} строк')
print(f'agg_cell_space:   {len(agg_cell):,} ячеек')

# Карточка качества координат
total_dtp    = len(dtp)
enriched     = feat_spatial['dtp_id'].isin(dtp['dtp_id']).sum()
valid_coords = feat_spatial['coord_valid'].sum() if 'coord_valid' in feat_spatial.columns else None

print(f'\nКарточка качества геоданных:')
print(f'  Всего ДТП в периоде:      {total_dtp:,}')
print(f'  Обогащено пространством:  {enriched:,} ({enriched/total_dtp*100:.1f}%)')
if valid_coords is not None:
    print(f'  Валидных координат:       {int(valid_coords):,} ({valid_coords/len(feat_spatial)*100:.1f}%)')

# ── 2. Джойн ДТП + пространственные признаки ─────────────────────────────────
dtp_sp = dtp.merge(feat_spatial, on='dtp_id', how='inner')
print(f'\nПосле джойна dtp × feat_spatial: {len(dtp_sp):,} строк')

# ── 3. Агрегация по cell_id × year ───────────────────────────────────────────
spatial_agg = (
    dtp_sp.groupby(['cell_id', 'year'], dropna=False)
    .agg(
        dtp_count             = ('dtp_id',                   'count'),
        dead_total            = ('dead_count',                'sum'),
        injured_total         = ('injured_count',             'sum'),
        dead_children_total   = ('dead_children_count',       'sum'),
        pct_fatal             = ('severity',                  lambda x: (x == 'Гибель').mean()),
        pct_injured           = ('severity',                  lambda x: (x == 'Ранение').mean()),
        pct_speed_viol        = ('has_speed_violation',       'mean'),
        pct_priority_viol     = ('has_priority_violation',    'mean'),
 
        pct_pedestrian        = ('has_pedestrian_conflict',   'mean'),
        pct_bad_road          = ('has_bad_road_condition',    'mean'),
        pct_night             = ('is_night',                  'mean'),
        pct_weekend           = ('is_weekend',                'mean'),
        pct_rush_hour         = ('is_rush_hour',              'mean'),
        pct_at_intersection   = ('is_intersection',           'mean'),
        pct_on_open_road      = ('is_open_road',              'mean'),
        pct_has_camera        = ('has_camera',                'mean'),
        pct_has_crosswalk_pt  = ('has_crosswalk',             'mean'),
        pct_has_traffic_light = ('has_traffic_light',         'mean'),
        pct_has_bus_stop      = ('has_bus_stop',              'mean'),
     
        avg_lane_width_m      = ('lane_width_m',              'mean'),
        avg_dist_intersection = ('distance_to_intersection_m','mean'),
        avg_dist_crosswalk    = ('distance_to_crosswalk_m',   'mean'),
        avg_dist_bus_stop     = ('distance_to_bus_stop_m',    'mean'),
        avg_poi_100m          = ('poi_count_100m',            'mean'),
        avg_poi_300m          = ('poi_count_300m',            'mean'),
    )
    .reset_index()
)

spatial_agg['mortality_rate'] = (
    spatial_agg['dead_total'] / spatial_agg['dtp_count'].clip(lower=1)
).round(4)

# ── 4. Присоединяем контекст ячейки из agg_cell_space ────────────────────────
cell_context_cols = [
    'cell_id', 'h3_res',
    'has_traffic_light', 'has_crosswalk', 'has_bus_stop',
    'is_intersection', 'is_open_road', 'has_camera',
    'traffic_lane_amount', 'lane_width_m', 'is_multilane',
    'distance_to_intersection_m', 'distance_to_bus_stop_m',
    'poi_count_100m', 'poi_count_300m',
    'crossing_density', 'stop_density',
    'speed_limit', 'road_class', 'landuse_type',
]
cell_context = agg_cell[[c for c in cell_context_cols if c in agg_cell.columns]].copy()
cell_context = cell_context.rename(columns={
    c: f'cell_{c}' for c in cell_context.columns if c != 'cell_id'
})

mart_spatial_risk = spatial_agg.merge(cell_context, on='cell_id', how='left')

mart_spatial_risk.to_csv('mart_spatial_risk.csv', index=False, encoding='utf-8-sig')
print(f'\nmart_spatial_risk: {len(mart_spatial_risk):,} строк × {mart_spatial_risk.shape[1]} колонок')
print('\nТоп-10 ячеек по числу ДТП:')
print(
    mart_spatial_risk.sort_values('dtp_count', ascending=False)
    [['cell_id', 'year', 'dtp_count', 'dead_total', 'injured_total',
      'pct_fatal', 'cell_road_class', 'cell_speed_limit', 'cell_landuse_type']]
    .head(10)
    .to_string(index=False)
)

feat_spatial_dtp: 24,552 строк
agg_cell_space:   1,986 ячеек

Карточка качества геоданных:
  Всего ДТП в периоде:      24,553
  Обогащено пространством:  24,552 (100.0%)
  Валидных координат:       24,552 (100.0%)

После джойна dtp × feat_spatial: 24,552 строк

mart_spatial_risk: 4,910 строк × 47 колонок

Топ-10 ячеек по числу ДТП:
        cell_id  year  dtp_count  dead_total  injured_total  pct_fatal cell_road_class  cell_speed_limit cell_landuse_type
8811aa7ad5fffff  2024         30           0             33   0.000000         footway              60.0       residential
8811aa7ad9fffff  2023         30           1             33   0.033333             NaN               NaN               NaN
8811aa7a83fffff  2024         29           0             31   0.000000         service              60.0       residential
8811aa7ad5fffff  2023         29           1             38   0.034483         footway              60.0       residential
8811aa7133fffff  2024         26           1       

In [ ]:
# ── agg_cell_space — агрегат по ячейкам (от исп. 2) ─────────────────────────
CELL_SPACE_AVAILABLE = os.path.exists(PATH_CELL_SPACE)

if CELL_SPACE_AVAILABLE:
    agg_cell_space = pd.read_csv(PATH_CELL_SPACE, low_memory=False)
    print(f'agg_cell_space загружен: {len(agg_cell_space):,} ячеек')

    # Обновляем mart_spatial_risk с H3-ячейками
    if 'cell_id' in dtp.columns and dtp['cell_id'].notna().any():
        mart_spatial_risk_h3 = (
            dtp.groupby(['year', 'cell_id'], dropna=False)
            .agg(
                dtp_count    = ('dtp_id', 'count'),
                dead_total   = ('dead_count', 'sum'),
                injured_total= ('injured_count', 'sum'),
                pct_bad_road = ('has_bad_road_condition', 'mean'),
                pct_speed    = ('has_speed_violation', 'mean'),
                pct_fatal    = ('severity', lambda x: (x == 'Гибель').mean()),
            )
            .reset_index()
            .merge(agg_cell_space, on='cell_id', how='left')
        )
        mart_spatial_risk_h3['mortality_rate'] = (
            mart_spatial_risk_h3['dead_total'] /
            mart_spatial_risk_h3['dtp_count'].clip(lower=1)).round(4)

        mart_spatial_risk_h3.to_csv('mart_spatial_risk_h3.csv', index=False, encoding='utf-8-sig')
        print(f'mart_spatial_risk_h3: {len(mart_spatial_risk_h3):,} строк')
else:
    print('⚠️  agg_cell_space.csv не найден. mart_spatial_risk_h3 не создаётся.')

---
## 9. Блок исполнителя 3 — Погода и световая среда [ЗАГЛУШКА]

⚠️ **Данные пока неполные.** Этот блок изолирован — он не влияет на остальные витрины.  
После получения `feat_weather_dtp.csv` нужно только запустить эти ячейки.

**Ожидаемые поля `feat_weather_dtp`:**

| Блок | Поля |
|---|---|
| Погода | `temp_c`, `feels_like_c`, `precipitation_type`, `precipitation_mm`, `snow_depth_cm`, `wind_speed_ms`, `humidity_pct`, `visibility_m` |
| Астрономия | `sunrise_ts`, `sunset_ts`, `daylight_minutes`, `is_twilight`, `is_dark`, `is_daylight` |
| Календарь | `day_of_week`, `is_weekend`, `is_holiday`, `holiday_name` |
| Индикаторы | `ice_risk`, `low_visibility_flag`, `bad_weather_flag`, `dark_without_lights_flag` |

In [ ]:
# WEATHER_AVAILABLE = os.path.exists(PATH_WEATHER)

# if WEATHER_AVAILABLE:
#     feat_weather = pd.read_csv(PATH_WEATHER, low_memory=False)
#     print(f'feat_weather_dtp загружен: {len(feat_weather):,} строк')

#     # Отчёт покрытия
#     coverage = feat_weather['dtp_id'].isin(dtp['dtp_id']).sum()
#     total    = len(dtp)
#     print(f'Покрытие погодой: {coverage:,} / {total:,} ДТП ({coverage/total*100:.1f}%)')

#     # Покрытие по отдельным полям
#     for col in ['temp_c', 'precipitation_mm', 'visibility_m', 'ice_risk',
#                 'low_visibility_flag', 'is_dark', 'is_holiday']:
#         if col in feat_weather.columns:
#             pct = feat_weather[col].notna().mean() * 100
#             print(f'  {col}: заполнено {pct:.1f}%')

#     # Приджойниваем
#     dtp_w = dtp.merge(feat_weather, on='dtp_id', how='left')
#     print('Погодные признаки добавлены.')
# else:
#     print('⚠️  feat_weather_dtp.csv не найден. Витрина mart_weather_context не создаётся.')
#     dtp_w = None

In [ ]:
# # ── mart_weather_context — только если данные есть ──────────────────────────
# if dtp_w is not None:
#     weather_group_cols = ['year', 'month', 'season']
#     if 'cell_id' in dtp_w.columns:
#         weather_group_cols.append('cell_id')

#     agg_dict = dict(
#         dtp_count   = ('dtp_id', 'count'),
#         dead_total  = ('dead_count', 'sum'),
#         injured_total = ('injured_count', 'sum'),
#     )
#     for col, label in [
#         ('ice_risk',             'pct_ice_risk'),
#         ('low_visibility_flag',  'pct_low_visibility'),
#         ('bad_weather_flag',     'pct_bad_weather'),
#         ('dark_without_lights_flag', 'pct_dark_no_lights'),
#         ('is_holiday',           'pct_holiday'),
#     ]:
#         if col in dtp_w.columns:
#             agg_dict[label] = (col, 'mean')

#     for col, label in [
#         ('temp_c',           'avg_temp_c'),
#         ('precipitation_mm', 'avg_precip_mm'),
#         ('visibility_m',     'avg_visibility_m'),
#         ('daylight_minutes', 'avg_daylight_min'),
#     ]:
#         if col in dtp_w.columns:
#             agg_dict[label] = (col, 'mean')

#     mart_weather_context = (
#         dtp_w.groupby(weather_group_cols, dropna=False)
#         .agg(**agg_dict)
#         .reset_index()
#     )
#     mart_weather_context.to_csv('mart_weather_context.csv', index=False, encoding='utf-8-sig')
#     print(f'mart_weather_context: {len(mart_weather_context):,} строк')

#     # weather_cell_time (ключ cell_id × date_bucket) — если есть cell_id и дата
#     if 'cell_id' in dtp_w.columns:
#         dtp_w['date_bucket'] = dtp_w['moment_date'].dt.to_period('M').astype(str)
#         weather_cell_time = (
#             dtp_w.groupby(['cell_id', 'date_bucket'], dropna=False)
#             .agg(dtp_count=('dtp_id', 'count'))
#             .reset_index()
#         )
#         for col in ['temp_c', 'precipitation_mm', 'ice_risk', 'low_visibility_flag']:
#             if col in dtp_w.columns:
#                 weather_cell_time = weather_cell_time.merge(
#                     dtp_w.groupby(['cell_id','date_bucket'])[col].mean().reset_index(),
#                     on=['cell_id','date_bucket'], how='left')
#         weather_cell_time.to_csv('weather_cell_time.csv', index=False, encoding='utf-8-sig')
#         print(f'weather_cell_time: {len(weather_cell_time):,} строк')
# else:
#     print('mart_weather_context пропускается — данных нет.')